# Music Annotation Project V7.8.2: Rebuilt-Builder Equivalence Audit

**Runtime:** CPU. Select **Runtime > Run all**.

This is the replacement equivalence audit for the rebuilt V7.8 independent-data builder. It tests the rebuilt pipeline on the original development pages and compares each stage to the durable V7.5 development caches/tables.

## What must match

1. authoritative observed/aligned page;
2. dilation-2 removal mask;
3. Telea reconstruction;
4. `geometry_t80` corrected image;
5. preserve/remove/ambiguous routing maps;
6. horizontal/vertical/compact support maps where cached;
7. connected-component IDs and geometry;
8. all shared V7.5 surrounding-area predictor fields;
9. V7.7 structural outcomes, using the V7.7 component dataset as the label reference;
10. byte-for-byte immutability outside the removal mask.

The audit is read-only. It never reads the independent V7.8 outcomes. A failure at an earlier stage is reported separately so downstream differences are not mistaken for independent-domain behaviour.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib


In [ ]:
import cv2,json,hashlib,math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime,timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
OUTPUT_ROOT=ROOT/'implementation_equivalence_audit_v7_8_2'
V75_RUN_OVERRIDE=None;V77_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None;MAX_PAGES=None
ATOL=1e-9;RTOL=1e-7
MIN_COMPONENT_AREA=20
PREDICTORS=['w','h','area','aspect_ratio','fill_ratio','touches_page_border','context_pixels','context_edge_density','context_dark_density','context_mean_intensity','context_intensity_std','maximum_mask_depth','horizontal_fraction','vertical_fraction','compact_fraction','preserve_fraction','remove_fraction','ambiguous_fraction','reconstruction_dependency_fraction']

def latest(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv'])
V77=Path(V77_RUN_OVERRIDE) if V77_RUN_OVERRIDE else latest(V77_ROOT,['tables/asymmetric_component_dataset.csv','v7_7_manifest.json'])
if V75 is None or V77 is None:raise RuntimeError('Complete V7.5 and V7.7 development runs are required.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN;T=OUT/'tables';D=OUT/'difference_images'
for p in [OUT,T,D]:p.mkdir(parents=True,exist_ok=True)
print('V7.5:',V75);print('V7.7:',V77);print('Output:',OUT)


## Load authoritative development artefacts


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
 with np.load(path) as z:return {k:z[k] for k in z.files}
def digest(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()+str(a.shape).encode()+str(a.dtype).encode()).hexdigest()
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0];assert np.array_equal(out[mask==0],original[mask==0]);return out
clean_map=imap(ROOT/'source'/'clean');observed_map=imap(ROOT/'aligned')
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
evidence=pd.read_csv(V75/'tables'/'surrounding_area_evidence_by_component.csv');v77=pd.read_csv(V77/'tables'/'asymmetric_component_dataset.csv')
for frame in [evidence]:
 c=next((x for x in ['method','candidate','geometry_candidate'] if x in frame.columns),None)
 if c is not None:frame.drop(frame[~frame[c].astype(str).str.lower().str.contains('t80')].index,inplace=True)
pages=sorted(set(clean_map)&set(observed_map)&set(cache_lookup))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
print('Development pages:',len(pages))


## Rebuilt pipeline functions


In [ ]:
# The audit deliberately starts from the cached V7.5 removal mask and Telea arrays.
# This isolates V7.5 geometry/evidence equivalence from V7.1 cache reproduction.
def background_surface(observed,mask,sigma=45.0):
 visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32);num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma);return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps(observed,mask,band=4):
 dark=(observed<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=stats[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
 k=np.ones((2*band+1,2*band+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0
def geometry_t80(observed,mask,telea):
 paper=background_surface(observed,mask);norm=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0;preserve=m&(horizontal|vertical|compact)&(observed<200);remove=m&~preserve&(observed>=80)&(norm>=80);ambiguous=m&~preserve&~remove;result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve];return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact}
def features(observed,component,routes):
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;a=int(component.sum());w=x1-x0;h=y1-y0;k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(37,37));ring=(cv2.dilate(component.astype(np.uint8),k)>0)&~component;vals=observed[ring];edge=cv2.Canny(observed,70,170)>0;depth=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
 def f(name):return float(np.sum(routes[name]&component)/a)
 return {'w':w,'h':h,'area':a,'aspect_ratio':max(w,h)/max(min(w,h),1),'fill_ratio':a/max(w*h,1),'touches_page_border':bool(x0==0 or y0==0 or x1==observed.shape[1] or y1==observed.shape[0]),'context_pixels':int(ring.sum()),'context_edge_density':float(edge[ring].mean()) if ring.any() else 0.0,'context_dark_density':float((vals<190).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(depth[component].max()),'horizontal_fraction':f('horizontal'),'vertical_fraction':f('vertical'),'compact_fraction':f('compact'),'preserve_fraction':f('preserve'),'remove_fraction':f('remove'),'ambiguous_fraction':f('ambiguous'),'reconstruction_dependency_fraction':f('ambiguous')}


## Stage A: observed page, cached mask and Telea integrity


In [ ]:
stage_a=[];missing=[]
for page in pages:
 z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);obs=cv2.imread(str(observed_map[page]),0)
 required=['mask','telea','geometry_t80']
 absent=[x for x in required if x not in z]
 if absent:missing.append({'page':page,'missing':'|'.join(absent)});continue
 mask=z['mask'];telea=z['telea'];stage_a.append({'page':page,'observed_shape_ok':obs.shape==clean.shape,'mask_shape_ok':mask.shape==obs.shape,'telea_shape_ok':telea.shape==obs.shape,'telea_outside_mask_immutable':bool(np.array_equal(telea[mask==0],obs[mask==0])),'mask_pixels':int((mask>0).sum())})
pd.DataFrame(stage_a).to_csv(T/'stage_a_cache_integrity.csv',index=False);pd.DataFrame(missing).to_csv(T/'missing_arrays.csv',index=False)


## Stage B: geometry_t80 and routing equivalence


In [ ]:
geometry=[];routes=[];runtime={}
for page in pages:
 if page in {x['page'] for x in missing}:continue
 z=loadz(cache_lookup[page]);obs=cv2.imread(str(observed_map[page]),0);mask=z['mask'];telea=z['telea'];built,br=geometry_t80(obs,mask,telea);ref=z['geometry_t80'];geometry.append({'page':page,'exact':bool(np.array_equal(ref,built)),'different_pixels':int(np.sum(ref!=built)),'max_abs_difference':int(np.max(np.abs(ref.astype(np.int16)-built.astype(np.int16)))),'reference_hash':digest(ref),'rebuilt_hash':digest(built),'reference_outside_immutable':bool(np.array_equal(ref[mask==0],obs[mask==0])),'rebuilt_outside_immutable':bool(np.array_equal(built[mask==0],obs[mask==0]))})
 for key in ['preserve','remove','ambiguous','horizontal','vertical','compact']:
  rk=f'g80_{key}'
  if rk in z:
   a=z[rk]>0;b=br[key];routes.append({'page':page,'route':key,'tested':True,'exact':bool(np.array_equal(a,b)),'xor_pixels':int(np.logical_xor(a,b).sum()),'iou':float((a&b).sum()/max((a|b).sum(),1))})
  else:routes.append({'page':page,'route':key,'tested':False,'exact':False,'xor_pixels':np.nan,'iou':np.nan})
 diff=np.abs(ref.astype(np.int16)-built.astype(np.int16)).astype(np.uint8)
 if diff.any():cv2.imwrite(str(D/f'{page}_geometry_absdiff.png'),diff)
 runtime[page]=(obs,mask,built,br)
geometry_df=pd.DataFrame(geometry);route_df=pd.DataFrame(routes);geometry_df.to_csv(T/'stage_b_geometry.csv',index=False);route_df.to_csv(T/'stage_b_routing.csv',index=False)


## Stage C: connected components and V7.5 predictor equivalence


In [ ]:
components=[];pred=[]
for page,(obs,mask,built,br) in runtime.items():
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);ev=evidence[evidence.page.astype(str)==page]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=lab==cid;f=features(obs,comp,br);ref=ev[ev.component.astype(int)==cid];components.append({'page':page,'component':cid,'reference_row_count':len(ref),'x':int(stats[cid,0]),'y':int(stats[cid,1]),'w_cc':int(stats[cid,2]),'h_cc':int(stats[cid,3]),'area_cc':int(stats[cid,4])})
  for field in PREDICTORS:
   rv=ref.iloc[0][field] if len(ref)==1 and field in ref.columns else np.nan;bv=f[field]
   if isinstance(bv,(bool,np.bool_)):match=pd.notna(rv) and bool(rv)==bool(bv);delta=0 if match else 1
   else:match=pd.notna(rv) and bool(np.isclose(float(rv),float(bv),atol=ATOL,rtol=RTOL,equal_nan=True));delta=abs(float(rv)-float(bv)) if pd.notna(rv) else np.nan
   pred.append({'page':page,'component':cid,'field':field,'reference_value':rv,'rebuilt_value':bv,'match':match,'absolute_difference':delta})
component_df=pd.DataFrame(components);pred_df=pd.DataFrame(pred);component_df.to_csv(T/'stage_c_components.csv',index=False);pred_df.to_csv(T/'stage_c_predictors.csv',index=False)
summary=pred_df.groupby('field',as_index=False).agg(comparisons=('match','size'),mismatches=('match',lambda x:int((~x).sum())),max_absolute_difference=('absolute_difference','max'),mean_absolute_difference=('absolute_difference','mean')).sort_values('mismatches',ascending=False);summary.to_csv(T/'predictor_mismatch_summary.csv',index=False)


## Stage D: V7.7 label-reference alignment


In [ ]:
# This stage audits label identity by component against the actual V7.7 dataset rather than V7.5 truth diagnostics.
rows=[]
for page,(obs,mask,built,br) in runtime.items():
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);ref=v77[v77.page.astype(str)==page]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  r=ref[ref.component.astype(int)==cid];rows.append({'page':page,'component':cid,'v7_7_rows':len(r),'v7_7_structural_safe':r.iloc[0].structural_safe if len(r)==1 and 'structural_safe' in r else np.nan,'v7_7_failure_reasons':r.iloc[0].structural_failure_reasons if len(r)==1 and 'structural_failure_reasons' in r else np.nan})
labels=pd.DataFrame(rows);labels.to_csv(T/'stage_d_v7_7_label_alignment.csv',index=False)


## Stage-by-stage fail-closed verdict


In [ ]:
a=pd.DataFrame(stage_a);all_route_tested=bool(len(route_df) and route_df.tested.all());checks={'authoritative_arrays_complete':len(missing)==0,'stage_a_shapes_and_telea':bool(len(a) and a[['observed_shape_ok','mask_shape_ok','telea_shape_ok','telea_outside_mask_immutable']].all().all()),'stage_b_geometry_exact':bool(len(geometry_df) and geometry_df.exact.all()),'stage_b_routing_arrays_available':all_route_tested,'stage_b_routing_exact':bool(all_route_tested and route_df.exact.all()),'stage_c_component_reference_unique':bool(len(component_df) and (component_df.reference_row_count==1).all()),'stage_c_predictors_exact':bool(len(pred_df) and pred_df.match.all()),'stage_d_v7_7_component_alignment':bool(len(labels) and (labels.v7_7_rows==1).all())}
order=['authoritative_arrays_complete','stage_a_shapes_and_telea','stage_b_geometry_exact','stage_b_routing_arrays_available','stage_b_routing_exact','stage_c_component_reference_unique','stage_c_predictors_exact','stage_d_v7_7_component_alignment'];first_failure=next((x for x in order if not checks[x]),None);verdict='PASS' if first_failure is None else 'FAIL'
result=pd.DataFrame([{'verdict':verdict,'first_failed_stage':first_failure,**checks,'pages':len(runtime),'components':len(component_df),'geometry_different_pixels':int(geometry_df.different_pixels.sum()) if len(geometry_df) else np.nan,'routing_xor_pixels':int(route_df.xor_pixels.fillna(0).sum()) if len(route_df) else np.nan,'predictor_mismatches':int((~pred_df.match).sum()) if len(pred_df) else np.nan}]);result.to_csv(T/'audit_verdict.csv',index=False);display(result)


## Charts and report


In [ ]:
if len(geometry_df):
 q=geometry_df.sort_values('different_pixels');plt.figure(figsize=(12,max(5,.25*len(q))));plt.barh(q.page,q.different_pixels);plt.xlabel('Different geometry_t80 pixels');plt.title('Rebuilt V7.8 versus V7.5 geometry');plt.tight_layout();plt.savefig(OUT/'geometry_difference_by_page.png',dpi=180);plt.close()
if len(route_df):
 q=route_df.groupby('route',as_index=False).xor_pixels.sum().sort_values('xor_pixels');plt.figure(figsize=(9,5));plt.barh(q.route,q.xor_pixels);plt.xlabel('Routing XOR pixels');plt.title('Rebuilt V7.8 versus V7.5 routing');plt.tight_layout();plt.savefig(OUT/'routing_difference_by_class.png',dpi=180);plt.close()
if len(summary):
 q=summary.sort_values('mismatches');plt.figure(figsize=(10,7));plt.barh(q.field,q.mismatches);plt.xlabel('Component-field mismatches');plt.title('Rebuilt V7.8 predictor mismatches');plt.tight_layout();plt.savefig(OUT/'predictor_mismatches.png',dpi=180);plt.close()
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.8.2','v7_5_reference':str(V75),'v7_7_reference':str(V77),'verdict':verdict,'first_failed_stage':first_failure,'checks':checks,'pages_audited':len(runtime),'components_audited':len(component_df),'read_only':True,'independent_data_read':False,'independent_validation_may_proceed':bool(verdict=='PASS')};(OUT/'v7_8_2_manifest.json').write_text(json.dumps(manifest_out,indent=2));lines=['V7.8.2 REBUILT-BUILDER EQUIVALENCE AUDIT','',f'VERDICT: {verdict}',f'First failed stage: {first_failure}',f'Pages: {len(runtime)}',f'Components: {len(component_df)}',f'Geometry differences: {int(geometry_df.different_pixels.sum()) if len(geometry_df) else "not tested"}',f'Routing XOR pixels: {int(route_df.xor_pixels.fillna(0).sum()) if len(route_df) else "not tested"}',f'Predictor mismatches: {int((~pred_df.match).sum()) if len(pred_df) else "not tested"}','','CHECKS']+[f'- {k}: {v}' for k,v in checks.items()]+['','PASS means the rebuilt geometry/evidence pipeline is computationally equivalent on the available development artefacts. It does not itself validate the independent safety policy.','FAIL means fix only the first failed development-equivalence stage before creating a new independent lock.'];(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines));print('Output:',OUT)
